# CoRAL-Sep Stage 2: Universal Adapter Baseline

Train a single adapter on **all** conditions combined.
Decision criterion: if universal adapter is within **0.5 dB SI-SDRi** of the
best per-adapter on its primary condition → adopt universal (simpler).

Run **after** Stage 1 adapters are complete.
Expected time: ~6 hours on T4.

In [ ]:
DATA_ROOT      = "/kaggle/input/calmsep-8k"
CHECKPOINT_DIR = "/kaggle/working/adapters"
STAGE1_DIR     = "/kaggle/working/adapters"  # where Stage 1 adapters were saved
EPOCHS         = 40
BATCH_SIZE     = 8
LR             = 1e-4
DEVICE         = "cuda"
REPO_PATH      = "/kaggle/input/calmsep-code"
NOISE_DIR      = "/kaggle/input/calmsep-8k/noise"
RIR_BANK       = "/kaggle/input/calmsep-8k/rirs/bank.json"
BUT_DIR        = "/kaggle/input/calmsep-8k/but-reverbdb"


In [ ]:
import subprocess
import sys

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

cmd = [
    sys.executable, "-m", "train.stage2_universal",
    "--data-root", DATA_ROOT,
    "--checkpoint-dir", CHECKPOINT_DIR,
    "--stage1-dir", STAGE1_DIR,
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--lr", str(LR),
    "--device", DEVICE,
    "--noise-dir", NOISE_DIR,
    "--rir-bank",  RIR_BANK,
    "--bf16",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=REPO_PATH)

In [ ]:
# ── Decision: universal vs per-adapter ────────────────────────────────────────
# Stage 2 training script prints the comparison table.
# Check the log above: if universal is within 0.5 dB on each primary condition,
# use universal (simpler). Otherwise proceed with 3 per-adapter + Stage 3.
import os

print("Universal adapter:", os.path.exists(os.path.join(CHECKPOINT_DIR, "universal_adapter.pt")))